# Fine-Tuning Phi-2 for Support Ticket Classification (QLoRA)

This notebook fine-tunes Microsoft's Phi-2 (2.7B parameters) to classify customer support tickets into 4 categories: **billing**, **technical**, **account**, **general**.

**Technique:** QLoRA — quantizes the base model to 4-bit and trains tiny adapter layers (~1% of total weights), so the entire process fits in Colab's free T4 GPU.

## Before you start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Have your HuggingFace token ready (from https://huggingface.co/settings/tokens)

## Step 1: Install dependencies

In [1]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl huggingface_hub scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.4 MB/s eta 0:00:00


## Step 2: Login to HuggingFace
This lets us push the fine-tuned model to your HuggingFace profile.

In [4]:
from huggingface_hub import notebook_login
notebook_login()

## Step 3: Verify GPU is available

In [5]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")

GPU available: True
GPU: Tesla T4
Memory: 15.6 GB


## Step 4: Generate the training dataset
We generate 800 synthetic support tickets (200 per category) right here in Colab.

In [6]:
import random
import re
from collections import Counter

TEMPLATES = {
    "billing": [
        "I was charged ${amount} twice on my credit card for order #{order_id}.",
        "My invoice from {month} shows an incorrect amount. I should have been charged ${amount} not ${wrong_amount}.",
        "I need a refund for my last payment of ${amount}. The service was not as described.",
        "Why was I charged ${amount} when my plan is supposed to be ${plan_amount}/month?",
        "I cancelled my subscription but I'm still being charged ${amount} every month.",
        "Can you explain the extra ${amount} fee on my latest bill?",
        "I need to update my payment method from Visa to Mastercard.",
        "My payment of ${amount} failed but the money was deducted from my bank account.",
        "I want to downgrade my plan from premium (${amount}/mo) to basic (${plan_amount}/mo).",
        "Please send me a receipt for my payment of ${amount} on {month} {day}.",
        "I'm seeing a charge of ${amount} that I don't recognize on my statement.",
        "Can I get a discount if I pay annually instead of monthly?",
        "My promo code {promo} isn't applying the discount at checkout.",
        "I was promised a {discount}% discount but I'm being charged full price.",
        "How do I cancel auto-renewal? I don't want to be charged again.",
    ],
    "technical": [
        "The app crashes every time I try to {action}. I'm on {os} version {version}.",
        "I'm getting error code {error_code} when trying to {action}.",
        "The {feature} feature has been loading for over {minutes} minutes.",
        "My data isn't syncing between my phone and desktop. Last sync was {days} days ago.",
        "The API returns a {status_code} error when I send a POST request to /api/{endpoint}.",
        "Integration with {service} stopped working after the latest update.",
        "The export function generates a corrupted {file_type} file every time.",
        "Page load times have increased from {fast}s to {slow}s since last week.",
        "The search function returns no results even for exact matches.",
        "Two-factor authentication is not sending the SMS code to my phone.",
        "The {feature} button is grayed out and I can't click it.",
        "I can't upload files larger than {size}MB even though the limit should be {limit}MB.",
        "The dashboard shows incorrect metrics — the numbers don't match the raw data.",
        "Getting a blank white screen after logging in on {browser}.",
        "The webhook endpoint isn't receiving any events since {month} {day}.",
    ],
    "account": [
        "I can't log in to my account. It says my password is incorrect but I just reset it.",
        "I need to change the email address on my account from {old_email} to {new_email}.",
        "My account was locked after too many failed login attempts.",
        "I want to delete my account and all associated data permanently.",
        "How do I enable two-factor authentication on my account?",
        "I need to transfer ownership of my team account to another admin.",
        "My account shows I'm on the free plan but I upgraded to premium last week.",
        "I can't access the admin dashboard even though I have admin permissions.",
        "Someone may have accessed my account without authorization.",
        "I need to merge my two accounts ({old_email} and {new_email}) into one.",
        "My profile picture won't update — it keeps showing the old one.",
        "How do I add team members to my organization account?",
        "I forgot which email I used to sign up. Can you help me find my account?",
        "My account settings keep reverting to defaults after I save changes.",
        "I need a copy of all data associated with my account for compliance purposes.",
    ],
    "general": [
        "What are your business hours for customer support?",
        "Do you offer any enterprise plans for companies with {num}+ employees?",
        "I'm interested in your product. Can you tell me how it compares to {competitor}?",
        "Where can I find documentation for the {feature} feature?",
        "Is there a mobile app available for {os}?",
        "What's your uptime SLA for the enterprise plan?",
        "Do you have any case studies from companies in the {industry} industry?",
        "I'd like to schedule a demo with your sales team.",
        "What's on your product roadmap for {quarter} {year}?",
        "Do you support integration with {service}?",
        "What security certifications does your platform have?",
        "Is there a community forum or Slack channel for users?",
        "Can I use your service if my company is based in {country}?",
        "What's your data retention policy?",
        "Do you offer training or onboarding sessions for new teams?",
    ],
}

FILL_VALUES = {
    "amount": lambda: random.choice([9.99, 14.99, 29.99, 49.99, 79.99, 99.99, 149.99, 199.99]),
    "wrong_amount": lambda: random.choice([39.99, 59.99, 89.99, 119.99, 159.99]),
    "plan_amount": lambda: random.choice([9.99, 14.99, 19.99, 29.99]),
    "order_id": lambda: random.randint(100000, 999999),
    "month": lambda: random.choice(["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]),
    "day": lambda: random.randint(1, 28),
    "promo": lambda: random.choice(["SAVE20", "WELCOME10", "SPECIAL50", "NEWYEAR", "SUMMER25"]),
    "discount": lambda: random.choice([10, 15, 20, 25, 30, 50]),
    "action": lambda: random.choice(["upload a file", "save my project", "generate a report", "open the dashboard", "export data", "run a query"]),
    "os": lambda: random.choice(["iOS", "Android", "Windows", "macOS", "Linux"]),
    "version": lambda: f"{random.randint(10, 17)}.{random.randint(0, 9)}",
    "error_code": lambda: random.choice(["E-1001", "E-2003", "E-4012", "ERR-500", "TIMEOUT-408"]),
    "feature": lambda: random.choice(["dashboard", "analytics", "reporting", "billing", "notifications", "search", "calendar", "chat"]),
    "minutes": lambda: random.choice([5, 10, 15, 20, 30]),
    "days": lambda: random.randint(2, 14),
    "status_code": lambda: random.choice([400, 401, 403, 404, 500, 502, 503]),
    "endpoint": lambda: random.choice(["users", "orders", "reports", "analytics", "webhooks"]),
    "service": lambda: random.choice(["Slack", "Jira", "Salesforce", "Zapier", "Google Sheets"]),
    "file_type": lambda: random.choice(["CSV", "PDF", "Excel", "JSON"]),
    "fast": lambda: round(random.uniform(0.5, 1.5), 1),
    "slow": lambda: round(random.uniform(5.0, 15.0), 1),
    "browser": lambda: random.choice(["Chrome", "Firefox", "Safari", "Edge"]),
    "size": lambda: random.choice([10, 25, 50, 100]),
    "limit": lambda: random.choice([100, 250, 500]),
    "old_email": lambda: f"user{random.randint(1, 999)}@oldmail.com",
    "new_email": lambda: f"user{random.randint(1, 999)}@newmail.com",
    "num": lambda: random.choice([50, 100, 200, 500, 1000]),
    "competitor": lambda: random.choice(["Zendesk", "Intercom", "Freshdesk", "HubSpot"]),
    "industry": lambda: random.choice(["healthcare", "fintech", "education", "e-commerce", "SaaS"]),
    "quarter": lambda: random.choice(["Q1", "Q2", "Q3", "Q4"]),
    "year": lambda: random.choice([2025, 2026]),
    "country": lambda: random.choice(["Germany", "Japan", "Brazil", "India", "Australia"]),
}

def fill_template(template):
    def replacer(match):
        key = match.group(1)
        if key in FILL_VALUES:
            return str(FILL_VALUES[key]())
        return match.group(0)
    return re.sub(r"\{(\w+)\}", replacer, template)

def generate_dataset(samples_per_category=200, seed=42):
    random.seed(seed)
    records = []
    for label, templates in TEMPLATES.items():
        for _ in range(samples_per_category):
            template = random.choice(templates)
            text = fill_template(template)
            records.append({"text": text, "label": label})
    random.shuffle(records)
    return records

records = generate_dataset(samples_per_category=200)
dist = Counter(r["label"] for r in records)
print(f"Generated {len(records)} records")
print(f"Distribution: {dict(sorted(dist.items()))}")
print(f"\nSample: {records[0]}")

Generated 800 records
Distribution: {'account': 200, 'billing': 200, 'general': 200, 'technical': 200}

Sample: {'text': "How do I cancel auto-renewal? I don't want to be charged again.", 'label': 'billing'}


## Step 5: Prepare the dataset for training
Convert to HuggingFace Dataset format and split into train/test sets.

In [7]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

LABELS = ["account", "billing", "general", "technical"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

# Format each record as an instruction prompt
def format_prompt(record):
    return {
        "text": (
            f"Classify the following customer support ticket into one of these categories: "
            f"{', '.join(LABELS)}.\n\n"
            f"Ticket: {record['text']}\n\n"
            f"Category: {record['label']}"
        ),
        "label": LABEL2ID[record["label"]],
    }

formatted = [format_prompt(r) for r in records]

train_data, test_data = train_test_split(formatted, test_size=0.15, random_state=42,
                                         stratify=[r["label"] for r in formatted])

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

print(f"Train: {len(train_dataset)} samples")
print(f"Test:  {len(test_dataset)} samples")
print(f"\nExample prompt:\n{train_dataset[0]['text']}")

Train: 680 samples
Test:  120 samples

Example prompt:
Classify the following customer support ticket into one of these categories: account, billing, general, technical.

Ticket: Two-factor authentication is not sending the SMS code to my phone.

Category: technical


## Step 6: Load the base model with 4-bit quantization
This loads Phi-2 in 4-bit precision (QLoRA), reducing memory from ~11GB to ~3GB so it fits on a free T4 GPU.

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "microsoft/phi-2"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

print(f"\nModel loaded! Parameters: {model.num_parameters():,}")
print(f"Memory used: {model.get_memory_footprint() / 1e9:.2f} GB")

Loading tokenizer...


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Model loaded! Parameters: 2,779,683,840
Memory used: 1.78 GB


## Step 7: Configure LoRA adapters
Instead of training all 2.7B parameters, LoRA adds small trainable adapter matrices to specific layers. This is what makes fine-tuning possible on limited hardware.

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                          # rank of the adapter matrices
    lora_alpha=32,                 # scaling factor
    lora_dropout=0.05,             # dropout for regularization
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "dense"],  # which layers to adapt
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Trainable parameters: 10,485,760 / 2,790,169,600 (0.38%)


In [13]:
import torch

# Convert trainable LoRA parameters to float32.
# The 4-bit frozen base model stays quantized.
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.to(torch.float32)

trainable_dtypes = {
    str(parameter.dtype)
    for parameter in model.parameters()
    if parameter.requires_grad
}

print("Trainable parameter dtypes:", trainable_dtypes)

Trainable parameter dtypes: {'torch.float32'}


## Step 8: Train the model
This takes about 10-15 minutes on a T4 GPU.

In [15]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./results",

    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    logging_steps=10,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=30,
    lr_scheduler_type="cosine",

    # Disable AMP to avoid BF16/FP16 GradScaler conflict
    fp16=False,
    bf16=False,

    optim="paged_adamw_8bit",

    save_total_limit=2,
    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    max_length=256,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()
print("\nTraining complete")

Adding EOS to train dataset:   0%|          | 0/680 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/680 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/680 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/680 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/680 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Starting training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.899858,0.814572,0.688318,37331.000000,0.821897
100,0.707525,0.653837,0.638489,74692.000000,0.843226
129,0.663360,0.641975,0.637297,96162.000000,0.845163



Training complete


## Step 9: Test the fine-tuned model
Let's see how well it classifies tickets it hasn't seen before.

In [16]:
def classify_ticket(ticket_text):
    prompt = (
        f"Classify the following customer support ticket into one of these categories: "
        f"{', '.join(LABELS)}.\n\n"
        f"Ticket: {ticket_text}\n\n"
        f"Category:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    # Extract the category from the response
    response = response.strip().lower()
    for label in LABELS:
        if label in response:
            return label
    return response

# Test with examples the model hasn't seen
test_tickets = [
    ("I was charged $50 twice for the same order", "billing"),
    ("The app keeps crashing on my iPhone", "technical"),
    ("I can't reset my password", "account"),
    ("Do you have a student discount?", "general"),
    ("My subscription renewal failed but money was taken", "billing"),
    ("Error 500 when uploading large files", "technical"),
    ("How do I delete my account?", "account"),
    ("What integrations do you support?", "general"),
]

print("Testing fine-tuned model:\n")
correct = 0
for ticket, expected in test_tickets:
    predicted = classify_ticket(ticket)
    match = "✓" if predicted == expected else "✗"
    if predicted == expected:
        correct += 1
    print(f"  {match} '{ticket}'")
    print(f"    Expected: {expected} | Predicted: {predicted}\n")

print(f"Accuracy: {correct}/{len(test_tickets)} ({100*correct/len(test_tickets):.0f}%)")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Testing fine-tuned model:



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  ✓ 'I was charged $50 twice for the same order'
    Expected: billing | Predicted: billing

  ✓ 'The app keeps crashing on my iPhone'
    Expected: technical | Predicted: technical

  ✗ 'I can't reset my password'
    Expected: account | Predicted: technical

  ✗ 'Do you have a student discount?'
    Expected: general | Predicted: billing

  ✓ 'My subscription renewal failed but money was taken'
    Expected: billing | Predicted: billing

  ✓ 'Error 500 when uploading large files'
    Expected: technical | Predicted: technical

  ✓ 'How do I delete my account?'
    Expected: account | Predicted: account

  ✗ 'What integrations do you support?'
    Expected: general | Predicted: technical

Accuracy: 5/8 (62%)


## Step 10: Evaluate on the full test set

In [17]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

print("Evaluating on full test set (this takes a few minutes)...\n")

y_true = []
y_pred = []

for record in test_data:
    # Extract the original ticket text and label from the formatted prompt
    true_label = ID2LABEL[record["label"]]
    # Extract just the ticket text from the formatted prompt
    ticket_text = record["text"].split("Ticket: ")[1].split("\n\nCategory:")[0]

    predicted = classify_ticket(ticket_text)
    y_true.append(true_label)
    y_pred.append(predicted if predicted in LABELS else "unknown")

print("Classification Report:")
print(classification_report(y_true, y_pred, labels=LABELS))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred, labels=LABELS)
print(f"{'':>12} " + " ".join(f"{l:>10}" for l in LABELS))
for i, label in enumerate(LABELS):
    print(f"{label:>12} " + " ".join(f"{cm[i][j]:>10}" for j in range(len(LABELS))))

Evaluating on full test set (this takes a few minutes)...

Classification Report:
              precision    recall  f1-score   support

     account       1.00      0.67      0.80        30
     billing       0.97      1.00      0.98        30
     general       0.75      0.60      0.67        30
   technical       0.67      1.00      0.80        30

    accuracy                           0.82       120
   macro avg       0.85      0.82      0.81       120
weighted avg       0.85      0.82      0.81       120


Confusion Matrix:
                account    billing    general  technical
     account         20          0          6          4
     billing          0         30          0          0
     general          0          1         18         11
   technical          0          0          0         30


## Step 11: Push to HuggingFace Hub
This publishes your fine-tuned model so anyone can use it.

**Change the repo name below to match your HuggingFace username!**

In [19]:
# ============================================
# CHANGE THIS to your HuggingFace username!
HF_USERNAME = "SahanaReddy5"
# ============================================

REPO_NAME = f"{HF_USERNAME}/phi2-support-ticket-classifier"

print(f"Pushing model to {REPO_NAME}...")

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print(f"\nDone! Your model is live at: https://huggingface.co/{REPO_NAME}")

Pushing model to SahanaReddy5/phi2-support-ticket-classifier...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 28.7kB / 42.0MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]


Done! Your model is live at: https://huggingface.co/SahanaReddy5/phi2-support-ticket-classifier


## Step 12: Save training metrics for GitHub
Download these files and add them to your GitHub repo.

In [20]:
import json

metrics = {
    "model": MODEL_NAME,
    "technique": "QLoRA (4-bit)",
    "lora_r": 16,
    "lora_alpha": 32,
    "train_samples": len(train_dataset),
    "test_samples": len(test_dataset),
    "epochs": 3,
    "classification_report": classification_report(y_true, y_pred, labels=LABELS, output_dict=True),
}

with open("training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved training_metrics.json")
print("\nDownload this file and add it to your GitHub repo.")
print("In Colab: click the folder icon on the left -> right-click training_metrics.json -> Download")

Saved training_metrics.json

Download this file and add it to your GitHub repo.
In Colab: click the folder icon on the left -> right-click training_metrics.json -> Download
